In [1]:
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

source_dir = Path("C:\\D\\finlab_ml\\finlab_ml_course\\finlab_practice\\history\\tables")
target_dir = Path("export/parquet")
target_dir.mkdir(parents=True, exist_ok=True)

In [2]:
def prepare_for_parquet(df: pd.DataFrame, table_name: str) -> pd.DataFrame:
    """Normalize pandas object columns before Arrow schema inference.

    Several crawler responses contain both Python int and str values in the
    same object column (for example otc_cap_reduction.恢復買賣日期).
    Arrow may infer int64 from the first values and then fail on a string.
    The canonical `date` column remains datetime64; only object columns are
    converted to pandas' nullable string dtype.
    """
    df = df.reset_index()

    if "stock_id" not in df.columns or "date" not in df.columns:
        raise KeyError(f"{table_name}: expected stock_id and date columns")

    df["stock_id"] = df["stock_id"].astype("string")
    df["date"] = pd.to_datetime(df["date"], errors="raise")

    for column in df.select_dtypes(include=["object"]).columns:
        non_null = df[column].dropna()
        python_types = sorted({type(value).__name__ for value in non_null})
        if len(python_types) > 1:
            print(f"  mixed object column {column!r}: {python_types} -> string")
        df[column] = df[column].astype("string")

    return df.drop_duplicates(["stock_id", "date"], keep="last")


for source in sorted(source_dir.glob("*.pkl")):
    print("reading", source)
    df = pd.read_pickle(source)
    df = prepare_for_parquet(df, source.stem)

    target = target_dir / f"{source.stem}.parquet"
    # Spark cannot infer TIMESTAMP(NANOS). Write physical microseconds.
    # Reject real sub-microsecond values instead of silently truncating them.
    df.to_parquet(
        target, index=False, engine="pyarrow", compression="snappy",
        version="2.4", coerce_timestamps="us",
        allow_truncated_timestamps=False, use_deprecated_int96_timestamps=False,
    )

    # Fail immediately if the written file cannot be read back.
    parquet_file = pq.ParquetFile(target)
    if any("nanoseconds" in str(parquet_file.schema.column(i).logical_type).lower()
           for i in range(len(parquet_file.schema))):
        raise ValueError(f"{source.stem}: incompatible TIMESTAMP(NANOS) remains")
    parquet_columns = parquet_file.schema_arrow.names
    if parquet_columns != list(df.columns):
        raise ValueError(f"{source.stem}: Parquet columns changed after round trip")
    if parquet_file.metadata.num_rows != len(df):
        raise ValueError(f"{source.stem}: Parquet row count changed after round trip")

    print(f"written {target} ({len(df):,} rows)")
    del df

reading C:\D\finlab_ml\finlab_ml_course\finlab_practice\history\tables\balance_sheet.pkl
written export\parquet\balance_sheet.parquet (99,341 rows)
reading C:\D\finlab_ml\finlab_ml_course\finlab_practice\history\tables\bargin_report.pkl
written export\parquet\bargin_report.parquet (4,868,512 rows)
reading C:\D\finlab_ml\finlab_ml_course\finlab_practice\history\tables\benchmark.pkl
written export\parquet\benchmark.parquet (10,824,547 rows)
reading C:\D\finlab_ml\finlab_ml_course\finlab_practice\history\tables\cash_flows.pkl
written export\parquet\cash_flows.parquet (99,341 rows)
reading C:\D\finlab_ml\finlab_ml_course\finlab_practice\history\tables\income_sheet.pkl
written export\parquet\income_sheet.parquet (95,274 rows)
reading C:\D\finlab_ml\finlab_ml_course\finlab_practice\history\tables\income_sheet_cumulate.pkl
written export\parquet\income_sheet_cumulate.parquet (99,341 rows)
reading C:\D\finlab_ml\finlab_ml_course\finlab_practice\history\tables\monthly_report.pkl
written export\